# EDA - Silver layer

Since there is flexibility in how to deal with cleaning the data, sandbox is a good option for developing and verifying the cleaning logic (also efficient and lightweight) before it eventually goes into
`transformations/silver/marathon_obt.py`.

Each cleaning step is going to be tested here against the bronze table, and the number of
rows dropped at each stage is to be documented.

In [0]:
import sys

sys.path.append("..")

from utils.utils import (
    rename_columns_to_snake_case,
    parse_event_distance_value,
    parse_event_distance_unit,
    classify_event_type,
    parse_performance_to_seconds,
    parse_performance_to_km,
    is_valid_row,
    event_distance_in_km,
    event_time_in_hours,
    recompute_average_speed,
    is_plausible_speed,
    MAX_PLAUSIBLE_SPEED_KMH,
)
from pyspark.sql.functions import col, when

In [0]:
df = spark.read.table("marathos.bronze.raw_marathon")
df = rename_columns_to_snake_case(df)

rows_bronze = df.count()
print(f"Bronze rows: {rows_bronze:,}")
print(df.columns)

## Parse event distance into value + unit + type

Sanity check, regex catches all the formats and the classification is correct.

In [0]:
df = (
    df.withColumn("event_distance_value", parse_event_distance_value(col("event_distance_length")))
      .withColumn("event_distance_unit", parse_event_distance_unit(col("event_distance_length")))
      .withColumn("event_type", classify_event_type(col("event_distance_unit")))
)

df.select(
    "event_distance_length", "event_distance_value", "event_distance_unit", "event_type"
).limit(20).display()

In [0]:
df.groupBy("event_type", "event_distance_unit").count().orderBy(col("count").desc()).display()

## Parse performance into seconds (distance) or km (length)

In [0]:
df = (
    df.withColumn(
        "performance_seconds",
        when(col("event_type") == "distance", parse_performance_to_seconds(col("athlete_performance"))),
    )
    .withColumn(
        "performance_km",
        when(col("event_type") == "length", parse_performance_to_km(col("athlete_performance"))),
    )
)

df.select(
    "event_type", "athlete_performance", "performance_seconds", "performance_km"
).limit(20).display()

## Validity

distance race -> performance must be a time (h), length race -> performance must be a distance (km).

Rows failing the rule, to get dropped.

In [0]:
df = df.withColumn("is_valid", is_valid_row(col("event_type"), col("athlete_performance")))

df.groupBy("is_valid").count().display()

In [0]:
df_valid = df.filter(col("is_valid"))
rows_valid = df_valid.count()
print(f"Rows after validity filter: {rows_valid:,}  (dropped {rows_bronze - rows_valid:,})")

## Recompute average speed and check plausibility

The raw `athlete_average_speed` contains data-entry errors (10000 km/h).
Needs to recompute it from cleaned distance and time, keep only rows within a realistic range. And the raw column is kept for transparency.

In [0]:
df_valid = (
    df_valid.withColumn(
        "distance_km",
        event_distance_in_km(col("event_distance_unit"), col("event_distance_value"), col("performance_km")),
    )
    .withColumn(
        "time_h",
        event_time_in_hours(col("event_type"), col("performance_seconds"), col("event_distance_value")),
    )
    .withColumn("recomputed_speed_kmh", recompute_average_speed(col("distance_km"), col("time_h")))
)

df_valid.select(
    "event_type", "distance_km", "time_h", "recomputed_speed_kmh", "athlete_average_speed"
).limit(20).display()

In [0]:
df_valid = df_valid.withColumn("is_plausible", is_plausible_speed(col("recomputed_speed_kmh")))
df_valid.groupBy("is_plausible").count().display()

In [0]:
df_clean = df_valid.filter(col("is_plausible"))
rows_clean = df_clean.count()
print(f"Rows after speed filter: {rows_clean:,}  (dropped {rows_valid - rows_clean:,})")

## Summary - Data Quality

In [0]:
dropped_invalid = rows_bronze - rows_valid
dropped_speed = rows_valid - rows_clean
total_dropped = rows_bronze - rows_clean

print(f"Bronze rows:                  {rows_bronze:>12,}")
print(f"Dropped - invalid event/perf: {dropped_invalid:>12,}  ({dropped_invalid / rows_bronze * 100:5.2f}%)")
print(f"Dropped - implausible speed:  {dropped_speed:>12,}  ({dropped_speed / rows_bronze * 100:5.2f}%)")
print(f"{'-' * 45}")
print(f"Silver rows (kept):           {rows_clean:>12,}  ({rows_clean / rows_bronze * 100:5.2f}%)")
print(f"Total dropped:                {total_dropped:>12,}  ({total_dropped / rows_bronze * 100:5.2f}%)")

## Verifying the silver table

In [0]:
silver = spark.read.table("marathos.silver.marathon_obt")
silver.printSchema()
print(f"Silver rows: {silver.count():,}")
silver.limit(10).display()